# House Prices — подгруппа 1

**Задача:** по описанию дома предсказать цену (`SalePrice`).

**Наш стек (задан преподавателем):**
1. `LabelEncoder` — превратить текст в числа
2. `RandomForestRegressor` — модель, которая учится на примерах

**Идея в одном абзаце:** у нас есть таблица домов с ценой (train). Мы чистим данные, кодируем категории числами, учим лес деревьев предсказывать цену. Потом применяем ту же обработку к домам без цены (test) и получаем файл `submission.csv` для проверки.

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## 1. Данные

- `train.csv` — дома **с** известной ценой (на них учимся)
- `test.csv` — дома **без** цены (для них делаем прогноз)

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train["SalePrice"]          # то, что предсказываем
test_ids = test["Id"]           # понадобится для submission

X = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])

print(X.shape, X_test.shape)
X.head(3)

(1460, 79) (1459, 79)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal


## 2. Пропуски (NaN)

Модель не умеет работать с пустыми ячейками.

- числа → заполняем **медианой** (устойчивее среднего к выбросам)
- текст → ставим метку `"Missing"`

In [4]:
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median = X[col].median()
    X[col] = X[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("Missing").replace("nan", "Missing")
    X_test[col] = X_test[col].astype(str).fillna("Missing").replace("nan", "Missing")

print("пустых ячеек:", X.isna().sum().sum(), X_test.isna().sum().sum())

пустых ячеек: 0 0


## 3. LabelEncoder

Лес работает только с числами.  
`LabelEncoder` заменяет категории на целые: `RL → 0`, `RM → 1`, …

Учим энкодер на train+test вместе, чтобы в тесте не появилось «неизвестное» значение.

In [5]:
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

X.head(3)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,4,65.0,8450,1,1,3,3,0,4,...,0,0,3,2,1,0,2,2008,9,4
1,20,4,80.0,9600,1,1,3,3,0,2,...,0,0,3,2,1,0,5,2007,9,4
2,60,4,68.0,11250,1,1,0,3,0,4,...,0,0,3,2,1,0,9,2008,9,4


## 4. RandomForestRegressor

**Случайный лес** = много решающих деревьев. Каждое дерево чуть по-своему угадывает цену, итог — среднее.

Сначала проверяем себя на 20% train (validation). Метрика Kaggle — **RMSLE** (ошибка в логарифмах цен; меньше = лучше).

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(
    n_estimators=300,   # сколько деревьев
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_tr, y_tr)

pred = np.clip(model.predict(X_val), 0, None)
rmsle = np.sqrt(mean_squared_log_error(y_val, pred))
print(f"Validation RMSLE: {rmsle:.5f}")

## 5. Submission

Учимся на **всех** train-данных и пишем прогноз для test → `submission.csv`.

final = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
final.fit(X, y)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": np.clip(final.predict(X_test), 0, None),
})
submission.to_csv("submission.csv", index=False)
submission.head()